# Horizontal Pong — Multi-Agent Learning Curves

This notebook automatically loads all available training logs from `../artifacts/train_log_*.csv`
and compares learning dynamics across agents (up to 4+ agents if present).

It produces:
1. Smoothed reward learning curves.
2. Smoothed hits-per-episode curves (if `hits` exists).
3. Smoothed loss curves (`actor_loss`, `critic_loss`, `total_loss` when available).
4. Stability view (rolling reward std) and quick summary table.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Load all available train logs
artifacts_dir = Path("../artifacts")
log_paths = sorted(artifacts_dir.glob("train_log_*.csv"))

if not log_paths:
    raise FileNotFoundError("No train logs found in ../artifacts (expected train_log_*.csv)")

logs = {}
for p in log_paths:
    agent_name = p.stem.replace("train_log_", "")
    df = pd.read_csv(p)
    if "episode" not in df.columns:
        df["episode"] = np.arange(1, len(df) + 1)
    logs[agent_name] = df

print("Loaded logs:")
for name, df in logs.items():
    print(f"- {name:20s} rows={len(df):5d} columns={list(df.columns)}")

WINDOW = 100

In [ ]:
# Learning curves: reward (smoothed)
plt.figure(figsize=(11, 5))
for agent, df in logs.items():
    if "reward" not in df.columns:
        continue
    smooth = df["reward"].rolling(WINDOW, min_periods=1).mean()
    plt.plot(df["episode"], smooth, label=agent)

plt.title(f"Smoothed Reward Curves (window={WINDOW})")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Hits comparison (if available)
plt.figure(figsize=(11, 5))
plotted = False
for agent, df in logs.items():
    if "hits" not in df.columns:
        continue
    smooth = df["hits"].rolling(WINDOW, min_periods=1).mean()
    plt.plot(df["episode"], smooth, label=agent)
    plotted = True

if plotted:
    plt.title(f"Smoothed Hits per Episode (window={WINDOW})")
    plt.xlabel("Episode")
    plt.ylabel("Hits")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No 'hits' column found in loaded logs.")

In [ ]:
# Loss comparison (plots whatever losses are present)
loss_cols = ["actor_loss", "critic_loss", "total_loss"]
available_loss_cols = sorted({c for df in logs.values() for c in loss_cols if c in df.columns})

if not available_loss_cols:
    print("No known loss columns found in logs.")
else:
    n = len(available_loss_cols)
    fig, axes = plt.subplots(n, 1, figsize=(11, 4 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, available_loss_cols):
        for agent, df in logs.items():
            if col in df.columns:
                smooth = df[col].rolling(WINDOW, min_periods=1).mean()
                ax.plot(df["episode"], smooth, label=agent)
        ax.set_title(f"Smoothed {col} (window={WINDOW})")
        ax.set_ylabel(col)
        ax.legend()

    axes[-1].set_xlabel("Episode")
    plt.tight_layout()
    plt.show()

In [ ]:
# Stability + summary table
# 1) Reward stability via rolling std
plt.figure(figsize=(11, 5))
plotted = False
for agent, df in logs.items():
    if "reward" not in df.columns:
        continue
    reward_std = df["reward"].rolling(WINDOW, min_periods=5).std()
    plt.plot(df["episode"], reward_std, label=agent)
    plotted = True

if plotted:
    plt.title(f"Reward Volatility: rolling std (window={WINDOW})")
    plt.xlabel("Episode")
    plt.ylabel("Reward std")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No 'reward' column found in loaded logs.")

# 2) Quick comparison table using last WINDOW episodes
summary_rows = []
for agent, df in logs.items():
    row = {"agent": agent, "episodes": len(df)}

    for col in ["reward", "hits", "actor_loss", "critic_loss", "total_loss", "grad_norm"]:
        if col in df.columns and len(df) > 0:
            row[f"last{WINDOW}_{col}_mean"] = float(df[col].tail(WINDOW).mean())
            row[f"last{WINDOW}_{col}_std"] = float(df[col].tail(WINDOW).std(ddof=0))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("agent").reset_index(drop=True)
summary_df